# 01 · Collect

Downloads raw Steam responses into `data/raw` (reviews, news, store details, online players). Settings are in `configs/config.yaml`.

- **Plan first:** `plan()` asks Steam how many reviews each game × language has (one quick request each), so you see the real size and time before starting.
- **One progress bar for the whole run**, counted in requests, with an ETA.
- **Safe to stop and re-run.** Raw pages are saved as they arrive. Reviews resume where they stopped, and a later run only fetches new and edited reviews; other sources are simply fetched again. `02_parse_clean` deduplicates.
- **Long runs:** the kernel has to stay open. To run without Jupyter open, e.g. overnight in `tmux`:
  `uv run jupyter nbconvert --to notebook --execute --inplace notebooks/01_collect.ipynb`

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging

from src.collect.run import collect, plan
from src.config import CONFIG
from src.db import read
from src.db.connection import init_db

logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
init_db()

## What to collect

Change these to collect less (e.g. a pilot on two games). `None` means the value from `configs/config.yaml`.

In [4]:
SOURCES = "all"   # or e.g. ["reviews"]; all = store_details, reviews, news, player_counts
APPIDS = [2868840]     # e.g. [730, 570] or None
PER_LANG = 0  # reviews per game x language; None = config, 0 = all reviews

jobs = plan(SOURCES, APPIDS, PER_LANG)
requests = jobs["requests"].sum()
hours = requests / CONFIG.http.host_rps["store.steampowered.com"] / 3600
print(f"{len(jobs)} jobs, {requests:,} requests, about {hours:.1f} h")
jobs

counting reviews on Steam:   0%|          | 0/7 [00:00<?, ?job/s]

7 jobs, 2,137 requests, about 1.2 h


,source,appid,game,language,requests,steam_total,cap
0,store_details,2868840,Slay the Spire 2,NaN,4,<NA>,<NA>
1,reviews,2868840,Slay the Spire 2,english,898,89709,<NA>
2,reviews,2868840,Slay the Spire 2,russian,29,2887,<NA>
3,reviews,2868840,Slay the Spire 2,schinese,1143,114250,<NA>
4,reviews,2868840,Slay the Spire 2,tchinese,42,4164,<NA>
5,news,2868840,Slay the Spire 2,NaN,20,<NA>,<NA>
6,player_counts,2868840,Slay the Spire 2,NaN,1,<NA>,<NA>


## Run

In [ ]:
result = collect(jobs)
result.groupby(["source", "status"]).size()

  0%|          | 0/2137 [00:00<?, ?req/s]

In [ ]:
# Failed jobs (re-run this notebook to retry them)
result[result["status"] == "failed"]

## Review download state

One row per game and language, kept across runs.

In [ ]:
read.review_collection()

Next: `02_parse_clean.ipynb` loads the raw files into Postgres.